In [0]:
import gzip
import io
import uuid
from datetime import UTC, datetime

import requests
from pyspark.sql import Row
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("trigger_type", "manual")
dbutils.widgets.text("run_user", "anna")

trigger_type = dbutils.widgets.get("trigger_type")
run_user = dbutils.widgets.get("run_user")

In [0]:
storage_account = "annagaplanyandatabricks"
container = "pdb-data"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope="adls-scope", key="storage-key")
)

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/pdb_lab"
bronze_path = f"{base_path}/bronze"
system_path = f"{base_path}/system"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.bronze_fasta_seqres (
    run_id STRING,
    source_url STRING,
    source_etag STRING,
    ingested_at TIMESTAMP,
    entity_id STRING,
    pdb_code STRING,
    chain_id STRING,
    mol_type STRING,
    seq_length INT,
    description STRING,
    sequence STRING
) USING DELTA
LOCATION '{bronze_path}/bronze_fasta_seqres'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS pdb_lab.fasta_seqres_metadata (
    source_url STRING,
    last_etag STRING,
    last_modified STRING,
    downloaded_at TIMESTAMP,
    run_id STRING,
    row_count BIGINT
) USING DELTA
LOCATION '{system_path}/fasta_seqres_metadata'
""")

In [0]:
def start_run(trigger_type: str, run_user: str = "anna") -> str:
    run_id = str(uuid.uuid4())

    spark.sql(f"""
        INSERT INTO pdb_lab.history_pdb_actualizer
        SELECT
            '{run_id}' AS run_id,
            '{run_user}' AS run_user,
            current_timestamp() AS started_at,
            CAST(NULL AS TIMESTAMP) AS finished_at,
            'RUNNING' AS status,
            '{trigger_type}' AS trigger_type,
            array('pdb_seqres.txt.gz') AS input_ids,
            CAST(NULL AS STRING) AS error_message
    """)
    return run_id

In [0]:
def finish_run_success(run_id: str) -> None:
    spark.sql(f"""
        UPDATE pdb_lab.history_pdb_actualizer
        SET finished_at = current_timestamp(),
            status = 'SUCCESS'
        WHERE run_id = '{run_id}'
    """)

In [0]:
def finish_run_skipped(run_id: str, message: str) -> None:
    msg = message.replace("'", "''")
    spark.sql(f"""
        UPDATE pdb_lab.history_pdb_actualizer
        SET finished_at = current_timestamp(),
            status = 'SKIPPED',
            error_message = '{msg}'
        WHERE run_id = '{run_id}'
    """)

In [0]:
def finish_run_failed(run_id: str, error_message: str) -> None:
    err = error_message.replace("'", "''")
    spark.sql(f"""
        UPDATE pdb_lab.history_pdb_actualizer
        SET finished_at = current_timestamp(),
            status = 'FAILED',
            error_message = '{err}'
        WHERE run_id = '{run_id}'
    """)


In [0]:
def get_active_pdb_ids() -> list[str]:
    pdb_ids = [
        str(r["pdb_id"]).lower().strip()
        for r in spark.sql("""
            SELECT DISTINCT pdb_id
            FROM pdb_lab.config_pdb_actualizer
            WHERE is_active = true
        """).collect()
    ]

    if not pdb_ids:
        raise ValueError("No active pdb_ids found in pdb_lab.config_pdb_actualizer")

    for pdb_id in pdb_ids:
        if len(pdb_id) != 4 or not pdb_id.isalnum():
            raise ValueError(f"Invalid pdb_id in config_pdb_actualizer: {pdb_id}")

    return pdb_ids

In [0]:
def get_iteration_v1_scope_df(active_pdb_ids: list[str]):
    ids_sql = ", ".join([f"'{x}'" for x in active_pdb_ids])

    return spark.sql(f"""
        SELECT DISTINCT
            lower(pdb_code) AS pdb_code,
            entity_id
        FROM pdb_lab.bronze_entity
        WHERE lower(pdb_code) IN ({ids_sql})
    """)

In [0]:
FASTA_URL = "https://files.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt.gz"

In [0]:
def get_remote_file_metadata(url: str) -> dict:
    response = requests.head(url, timeout=120, allow_redirects=True)
    response.raise_for_status()

    return {
        "etag": response.headers.get("ETag"),
        "last_modified": response.headers.get("Last-Modified"),
        "content_length": response.headers.get("Content-Length"),
    }

In [0]:
def get_stored_etag() -> str | None:
    rows = spark.sql("""
        SELECT last_etag
        FROM pdb_lab.fasta_seqres_metadata
        ORDER BY downloaded_at DESC
        LIMIT 1
    """).collect()

    return rows[0]["last_etag"] if rows else None

In [0]:
def should_download(remote_etag: str | None, stored_etag: str | None) -> bool:
    if stored_etag is None:
        return True
    if remote_etag is None:
        return True
    return remote_etag != stored_etag

In [0]:
def download_gzip_text(url: str) -> str:
    response = requests.get(url, timeout=300)
    response.raise_for_status()

    with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as gz:
        return gz.read().decode("utf-8", errors="replace")

In [0]:
def parse_fasta_seqres(text: str) -> list[dict]:
    records = []
    current = None

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        if line.startswith(">"):
            if current is not None:
                records.append(current)

            header = line[1:]
            parts = header.split()

            if len(parts) < 3:
                raise ValueError(f"Invalid FASTA header: {line}")

            entity_id = parts[0]

            mol_part = parts[1]
            if not mol_part.startswith("mol:"):
                raise ValueError(f"Invalid mol part in header: {line}")
            mol_type = mol_part.split(":", 1)[1]

            length_part = parts[2]
            if not length_part.startswith("length:"):
                raise ValueError(f"Invalid length part in header: {line}")
            seq_length = int(length_part.split(":", 1)[1])

            description = " ".join(parts[3:]) if len(parts) > 3 else None

            if "_" in entity_id:
                pdb_code, chain_id = entity_id.split("_", 1)
            else:
                pdb_code = entity_id
                chain_id = None

            current = {
                "entity_id": entity_id,
                "pdb_code": pdb_code.lower(),
                "chain_id": chain_id,
                "mol_type": mol_type,
                "seq_length": seq_length,
                "description": description,
                "sequence": ""
            }
        else:
            if current is None:
                raise ValueError("Sequence line encountered before header")
            current["sequence"] += line

    if current is not None:
        records.append(current)

    return records

In [0]:
def update_fasta_metadata(
    source_url: str,
    etag: str | None,
    last_modified: str | None,
    run_id: str,
    row_count: int
) -> None:
    etag_sql = "NULL" if etag is None else f"'{etag.replace(\"'\", \"''\")}'"
    lm_sql = "NULL" if last_modified is None else f"'{last_modified.replace(\"'\", \"''\")}'"

    spark.sql(f"""
        INSERT INTO pdb_lab.fasta_seqres_metadata
        SELECT
            '{source_url}' AS source_url,
            {etag_sql} AS last_etag,
            {lm_sql} AS last_modified,
            current_timestamp() AS downloaded_at,
            '{run_id}' AS run_id,
            {row_count} AS row_count
    """)

In [0]:
def run_fasta_iteration_v2(trigger_type: str = "manual", run_user: str = "anna") -> str:
    run_id = start_run(trigger_type, run_user)

    try:
        active_pdb_ids = get_active_pdb_ids()
        print(f"Active pdb_ids from config_pdb_actualizer: {active_pdb_ids}")
        remote_meta = get_remote_file_metadata(FASTA_URL)
        remote_etag = remote_meta["etag"]
        remote_last_modified = remote_meta["last_modified"]
        stored_etag = get_stored_etag()

        print(f"Stored ETag: {stored_etag}")
        print(f"Remote ETag: {remote_etag}")

        if not should_download(remote_etag, stored_etag):
            message = "No source change detected by ETag; FASTA download skipped."
            print(message)
            finish_run_skipped(run_id, message)
            return run_id

        fasta_text = download_gzip_text(FASTA_URL)
        parsed_records = parse_fasta_seqres(fasta_text)

        if not parsed_records:
            raise ValueError("No FASTA records parsed from source file")

        rows = [
            Row(
                run_id=run_id,
                source_url=FASTA_URL,
                source_etag=remote_etag,
                ingested_at=datetime.now(UTC),
                entity_id=r["entity_id"],
                pdb_code=r["pdb_code"],
                chain_id=r["chain_id"],
                mol_type=r["mol_type"],
                seq_length=r["seq_length"],
                description=r["description"],
                sequence=r["sequence"],
            )
            for r in parsed_records
        ]

        df = spark.createDataFrame(rows)

        scope_df = get_iteration_v1_scope_df(active_pdb_ids)

        df_filtered = df.join(
            scope_df,
            on=["pdb_code", "entity_id"],
            how="inner"
        )

        row_count = df_filtered.count()
        print(f"Filtered FASTA row count: {row_count}")

        df_filtered.write.format("delta").mode("overwrite").saveAsTable("pdb_lab.bronze_fasta_seqres")

        update_fasta_metadata(
            source_url=FASTA_URL,
            etag=remote_etag,
            last_modified=remote_last_modified,
            run_id=run_id,
            row_count=row_count
        )

        finish_run_success(run_id)
        return run_id

    except Exception as e:
        finish_run_failed(run_id, str(e))
        raise


In [0]:
run_id = run_fasta_iteration_v2(trigger_type=trigger_type, run_user=run_user)